In [ ]:
!pip install -q imbalanced-learn lime shap seaborn scikit-learn xgboost lightgbm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 7.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
# ======================================================================
# TERS: Triangulated Explainability Reliability Score for
#       ACO-RF Alzheimer's Disease Prediction
#
# Backbone (BEFS + ACO-optimized Random Forest + train-only SMOTE,
# leakage-safe CV, bootstrap CI, McNemar's testing) follows the same
# family of methods published in:
#   Soladoye A.A. et al. (2025), "Enhancing Alzheimer's disease
#   prediction using random forest: A novel framework combining
#   backward feature elimination and ant colony optimization",
#   Current Research in Translational Medicine, 73(4), 103526.
# That paper reports ~95% accuracy on the same El-Kharoua-style
# dataset (2,149 instances / 34 features) using the same recipe.
# => The backbone below is treated as an ESTABLISHED baseline
#    architecture, NOT as this project's novelty claim. The novelty
#    claim of this script lives entirely in Sections 8-13 (the TERS
#    explainability-reliability framework). See the accompanying
#    research summary document for the full novelty assessment.


# !pip install -q imbalanced-learn lime shap seaborn scikit-learn xgboost lightgbm

from dataclasses import dataclass, field
from pathlib import Path
from typing import Tuple, Dict, Any, List, Optional
import itertools
import json
import warnings
import random

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
    RandomizedSearchCV,
)
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
)
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    brier_score_loss,
    log_loss,
    confusion_matrix,
    classification_report,
)
from sklearn.calibration import calibration_curve
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

import shap
from lime.lime_tabular import LimeTabularExplainer
import joblib

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False


# ================================================================
# 1. CONFIGURATION
# ================================================================
@dataclass
class Config:
    DATA_URL: str = (
        "https://raw.githubusercontent.com/"
        "SahandNamvar/Binary-classification-Alzheimer-disease/"
        "main/alzheimers_disease_data.csv"
    )
    TARGET: str = "Diagnosis"
    ID_COLUMNS: Tuple[str, ...] = ("PatientID", "DoctorInCharge")

    RANDOM_STATE: int = 42
    TEST_SIZE: float = 0.30

    CV_SPLITS: int = 5
    INNER_CV_SPLITS: int = 3

    FINAL_FEATURES: int = 26
    BEFS_CV_SPLITS: int = 3
    BEFS_N_JOBS: int = -1

    # ACO
    ACO_ITERATIONS: int = 20
    ANTS_PER_ITERATION: int = 10
    EVAPORATION_RATE: float = 0.40
    ALPHA: float = 1.0
    BETA: float = 2.0

    N_ESTIMATORS_VALUES: Tuple[int, ...] = (100, 200, 300, 500, 700, 1000)
    MAX_DEPTH_VALUES: Tuple[Any, ...] = (None, 5, 8, 10, 12, 15, 20, 25, 30)
    MIN_SAMPLES_SPLIT_VALUES: Tuple[int, ...] = (2, 3, 5, 8, 10, 15, 20)
    MIN_SAMPLES_LEAF_VALUES: Tuple[int, ...] = (1, 2, 3, 4, 5, 8, 10)
    MAX_FEATURES_VALUES: Tuple[Any, ...] = ("sqrt", "log2", 0.4, 0.6, 0.8)
    CLASS_WEIGHT_VALUES: Tuple[Any, ...] = (None, "balanced", "balanced_subsample")

    W_ACC: float = 0.50
    W_F1: float = 0.30
    W_AUC: float = 0.20

    THRESHOLD_START: float = 0.25
    THRESHOLD_STOP: float = 0.76
    THRESHOLD_STEP: float = 0.01

    # XAI
    FUNCTIONAL_FEATURE: str = "FunctionalAssessment"
    LIME_NUM_FEATURES: int = 10
    LIME_NUM_SAMPLES: int = 3000
    XAI_MAX_INSTANCES: int = 300
    COUNTERFACTUAL_STEP: float = 0.05
    TOP_K_LOCAL_FEATURES: int = 5          # NEW: generalise XDCR beyond 1 feature

    # NEW: reliability-framework knobs
    N_BOOTSTRAP_CI: int = 1000             # predictive-metric bootstrap CIs
    N_BOOTSTRAP_STABILITY: int = 30        # explanation-stability retrains (SHAP heavy)
    N_SEED_REPEATS: int = 10               # random-seed CV stability
    ECE_BINS: int = 10
    DCA_THRESHOLDS: Tuple[float, float, float] = (0.01, 0.99, 0.01)  # start, stop, step
    TOP_K_CONSENSUS: Tuple[int, ...] = (5, 10)
    NESTED_CV_OUTER_SPLITS: int = 5
    NESTED_CV_USE_FULL_ACO: bool = False   # True = re-run ACO per outer fold (very slow)
    NESTED_CV_RANDOM_ITER: int = 40        # RandomizedSearchCV budget when not using full ACO

    TERS_WEIGHTS: Dict[str, float] = field(
        default_factory=lambda: {"global": 1 / 3, "stability": 1 / 3, "local": 1 / 3}
    )

    OUTPUT_DIR: str = "outputs_ters"


CFG = Config()
OUTPUT_DIR = Path(CFG.OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(CFG.RANDOM_STATE)
random.seed(CFG.RANDOM_STATE)

# Feature -> clinical domain map used by the domain-attribution analysis (Sec. 11).
# Engineered interaction features inherit the domain of their cognitive parent.
DOMAIN_MAP = {
    "cognitive_functional": [
        "MMSE", "FunctionalAssessment", "MemoryComplaints", "BehavioralProblems",
        "ADL", "Confusion", "Disorientation", "PersonalityChanges",
        "DifficultyCompletingTasks", "Forgetfulness",
        "Cognitive_Function_Interaction", "Cognitive_ADL_Interaction",
        "Functional_ADL_Interaction",
    ],
    "cardiometabolic": [
        "BMI", "SystolicBP", "DiastolicBP", "CholesterolTotal", "CholesterolLDL",
        "CholesterolHDL", "CholesterolTriglycerides", "CardiovascularDisease",
        "Diabetes", "Hypertension", "PulsePressure", "Triglyceride_HDL_Ratio",
    ],
    "demographic_social": ["Age", "Gender", "Ethnicity", "EducationLevel"],
    "lifestyle": ["Smoking", "AlcoholConsumption", "PhysicalActivity",
                  "DietQuality", "SleepQuality"],
    "risk_history": ["FamilyHistoryAlzheimers", "Depression", "HeadInjury"],
}


def feature_domain(feature: str) -> str:
    for domain, feats in DOMAIN_MAP.items():
        if feature in feats:
            return domain
    return "other"


# ================================================================
# 2. LOAD DATA
# ================================================================
def load_dataset() -> pd.DataFrame:
    print("=" * 80)
    print("LOADING ALZHEIMER DATASET")
    print("=" * 80)
    df = pd.read_csv(CFG.DATA_URL)
    print("Shape:", df.shape)
    print(df[CFG.TARGET].value_counts().sort_index())
    return df


# ================================================================
# 3. PREPARE DATA (unchanged from original, controlled FE only)
# ================================================================
def prepare_data(df: pd.DataFrame):
    df = df.copy()
    drop_cols = [c for c in CFG.ID_COLUMNS if c in df.columns]
    if drop_cols:
        df = df.drop(columns=drop_cols)

    if CFG.TARGET not in df.columns:
        raise ValueError(f"Target '{CFG.TARGET}' not found.")

    X = df.drop(columns=[CFG.TARGET]).copy()
    y = df[CFG.TARGET].astype(int).copy()

    if X.select_dtypes(exclude=np.number).shape[1] > 0:
        bad = X.select_dtypes(exclude=np.number).columns.tolist()
        raise ValueError(f"Non-numeric columns found: {bad}")
    if X.isnull().sum().sum() > 0:
        raise ValueError("Missing values found. Clean/impute before training.")

    if {"SystolicBP", "DiastolicBP"}.issubset(X.columns):
        X["PulsePressure"] = X["SystolicBP"] - X["DiastolicBP"]
    if {"CholesterolTriglycerides", "CholesterolHDL"}.issubset(X.columns):
        X["Triglyceride_HDL_Ratio"] = X["CholesterolTriglycerides"] / (X["CholesterolHDL"] + 1e-6)
    if {"MMSE", "FunctionalAssessment"}.issubset(X.columns):
        X["Cognitive_Function_Interaction"] = X["MMSE"] * X["FunctionalAssessment"]
    if {"MMSE", "ADL"}.issubset(X.columns):
        X["Cognitive_ADL_Interaction"] = X["MMSE"] * X["ADL"]
    if {"FunctionalAssessment", "ADL"}.issubset(X.columns):
        X["Functional_ADL_Interaction"] = X["FunctionalAssessment"] * X["ADL"]

    print("Predictor count after candidate feature engineering:", X.shape[1])
    return X, y


def split_data(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=CFG.TEST_SIZE, stratify=y, random_state=CFG.RANDOM_STATE,
    )
    print("Train:", X_train.shape, " Test:", X_test.shape)
    return X_train, X_test, y_train, y_test


# ================================================================
# 4. LEAKAGE-SAFE PIPELINE BUILDERS
# ================================================================
def build_cv_pipeline(params: Dict[str, Any], use_smote: bool = True) -> ImbPipeline:
    rf = RandomForestClassifier(random_state=CFG.RANDOM_STATE, n_jobs=-1, **params)
    steps = [("scaler", MinMaxScaler())]
    if use_smote:
        steps.append(("smote", SMOTE(random_state=CFG.RANDOM_STATE, k_neighbors=5)))
    steps.append(("rf", rf))
    return ImbPipeline(steps=steps)


# ================================================================
# 5. BACKWARD FEATURE ELIMINATION (unchanged)
# ================================================================
def run_befs(X_train: pd.DataFrame, y_train: pd.Series):
    print("\n" + "=" * 80)
    print("BACKWARD FEATURE ELIMINATION (LEAKAGE-SAFE)")
    print("=" * 80)
    from sklearn.feature_selection import SequentialFeatureSelector

    base_rf = RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_split=2, min_samples_leaf=1,
        max_features="sqrt", class_weight="balanced",
        random_state=CFG.RANDOM_STATE, n_jobs=-1,
    )
    estimator = build_cv_pipeline({
        "n_estimators": 300, "max_depth": None, "min_samples_split": 2,
        "min_samples_leaf": 1, "max_features": "sqrt", "class_weight": "balanced",
    })
    cv = StratifiedKFold(n_splits=CFG.BEFS_CV_SPLITS, shuffle=True, random_state=CFG.RANDOM_STATE)
    n_select = min(CFG.FINAL_FEATURES, X_train.shape[1])

    selector = SequentialFeatureSelector(
        estimator=estimator, n_features_to_select=n_select, direction="backward",
        scoring="accuracy", cv=cv, n_jobs=CFG.BEFS_N_JOBS,
    )
    selector.fit(X_train, y_train)
    selected = X_train.columns[selector.get_support()].tolist()
    print(f"Selected {len(selected)} features.")
    return selected, selector


# ================================================================
# 6. ACO-RF HYPERPARAMETER OPTIMIZATION (unchanged - established method,
#    see Soladoye et al. 2025 for the same recipe)
# ================================================================
class ACORandomForest:
    def __init__(self):
        self.param_spaces = {
            "n_estimators": list(CFG.N_ESTIMATORS_VALUES),
            "max_depth": list(CFG.MAX_DEPTH_VALUES),
            "min_samples_split": list(CFG.MIN_SAMPLES_SPLIT_VALUES),
            "min_samples_leaf": list(CFG.MIN_SAMPLES_LEAF_VALUES),
            "max_features": list(CFG.MAX_FEATURES_VALUES),
            "class_weight": list(CFG.CLASS_WEIGHT_VALUES),
        }
        self.pheromone = {k: np.ones(len(v), dtype=float) for k, v in self.param_spaces.items()}
        self.best_params = None
        self.best_score = -np.inf
        self.history = []

    @staticmethod
    def _normalize(v):
        v = np.asarray(v, dtype=float)
        if np.allclose(v.sum(), 0):
            return np.ones_like(v) / len(v)
        return v / v.sum()

    def _sample_index(self, key):
        tau = self.pheromone[key] ** CFG.ALPHA
        eta = np.ones_like(tau) ** CFG.BETA
        probs = self._normalize(tau * eta)
        return np.random.choice(len(probs), p=probs)

    def _sample_params(self):
        indices, params = {}, {}
        for key, values in self.param_spaces.items():
            idx = self._sample_index(key)
            indices[key] = idx
            params[key] = values[idx]
        return indices, params

    def _fitness(self, params, X, y, cv):
        pipeline = build_cv_pipeline(params)
        scores = cross_validate(
            pipeline, X, y, cv=cv,
            scoring={"acc": "accuracy", "f1": "f1", "auc": "roc_auc"},
            n_jobs=1, error_score="raise",
        )
        acc = float(np.mean(scores["test_acc"]))
        f1 = float(np.mean(scores["test_f1"]))
        auc = float(np.mean(scores["test_auc"]))
        composite = CFG.W_ACC * acc + CFG.W_F1 * f1 + CFG.W_AUC * auc
        return composite, acc, f1, auc

    def optimize(self, X, y, iterations=None, ants=None, verbose=True):
        iterations = iterations or CFG.ACO_ITERATIONS
        ants = ants or CFG.ANTS_PER_ITERATION
        cv = StratifiedKFold(n_splits=CFG.INNER_CV_SPLITS, shuffle=True, random_state=CFG.RANDOM_STATE)

        for iteration in range(iterations):
            candidates = []
            for _ in range(ants):
                indices, params = self._sample_params()
                composite, acc, f1, auc = self._fitness(params, X, y, cv)
                candidates.append((composite, acc, f1, auc, indices, params))
                if composite > self.best_score:
                    self.best_score = composite
                    self.best_params = params.copy()

            candidates.sort(key=lambda z: z[0], reverse=True)
            best = candidates[0]

            for key in self.pheromone:
                self.pheromone[key] *= (1.0 - CFG.EVAPORATION_RATE)
                self.pheromone[key] = np.maximum(self.pheromone[key], 1e-6)

            deposit = 1.0 / (1.0 - best[0] + 1e-6)
            for key, idx in best[4].items():
                self.pheromone[key][idx] += deposit

            self.history.append({
                "iteration": iteration + 1, "best_composite": best[0],
                "accuracy": best[1], "f1": best[2], "auc": best[3],
                "global_best": self.best_score,
            })
            if verbose:
                print(f"Iter {iteration+1:02d}/{iterations} | Acc={best[1]*100:.2f}% "
                      f"F1={best[2]*100:.2f}% AUC={best[3]:.4f} Global={self.best_score:.4f}")

        return self.best_params, pd.DataFrame(self.history)


# ================================================================
# 7. FINAL MODEL FIT / THRESHOLD TUNING (unchanged)
# ================================================================
def fit_final_model(X_train, y_train, X_test, selected_features, best_params, use_smote=True):
    scaler = MinMaxScaler()
    X_train_sel = X_train[selected_features].copy()
    X_test_sel = X_test[selected_features].copy()

    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_sel), columns=selected_features, index=X_train_sel.index)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test_sel), columns=selected_features, index=X_test_sel.index)

    if use_smote:
        smote = SMOTE(random_state=CFG.RANDOM_STATE, k_neighbors=5)
        X_resampled, y_resampled = smote.fit_resample(X_train_scaled, y_train)
    else:
        X_resampled, y_resampled = X_train_scaled, y_train

    model = RandomForestClassifier(random_state=CFG.RANDOM_STATE, n_jobs=-1, **best_params)
    model.fit(X_resampled, y_resampled)

    return model, scaler, X_train_scaled, X_test_scaled, X_resampled, y_resampled


def optimize_threshold(X_train, y_train, selected_features, best_params):
    X_sel = X_train[selected_features].copy()
    cv = StratifiedKFold(n_splits=CFG.CV_SPLITS, shuffle=True, random_state=CFG.RANDOM_STATE)
    pipeline = build_cv_pipeline(best_params)
    oof_prob = cross_val_predict(pipeline, X_sel, y_train, cv=cv, method="predict_proba", n_jobs=1)[:, 1]

    thresholds = np.arange(CFG.THRESHOLD_START, CFG.THRESHOLD_STOP + 1e-9, CFG.THRESHOLD_STEP)
    records, best_score, best_threshold = [], -np.inf, 0.50
    for t in thresholds:
        pred = (oof_prob >= t).astype(int)
        acc = accuracy_score(y_train, pred)
        f1 = f1_score(y_train, pred)
        score = 0.60 * acc + 0.40 * f1
        records.append((t, acc, f1, score))
        if score > best_score:
            best_score, best_threshold = score, float(t)

    threshold_df = pd.DataFrame(records, columns=["threshold", "accuracy", "f1", "objective"])
    threshold_df.to_csv(OUTPUT_DIR / "threshold_search.csv", index=False)
    return best_threshold, oof_prob, threshold_df


# ================================================================
# 8. NEW -- PLATT CALIBRATION FIT ON OOF PROBABILITIES (leakage-safe:
#    fit on training OOF predictions only, applied to test probabilities)
# ================================================================
def fit_platt_calibrator(oof_prob: np.ndarray, y_train: pd.Series) -> LogisticRegression:
    calibrator = LogisticRegression()
    calibrator.fit(oof_prob.reshape(-1, 1), y_train.values if hasattr(y_train, "values") else y_train)
    return calibrator


def apply_calibrator(calibrator: LogisticRegression, prob: np.ndarray) -> np.ndarray:
    return calibrator.predict_proba(prob.reshape(-1, 1))[:, 1]


def expected_calibration_error(y_true, y_prob, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(y_true)
    y_true = np.asarray(y_true)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (y_prob > lo) & (y_prob <= hi) if i > 0 else (y_prob >= lo) & (y_prob <= hi)
        if mask.sum() == 0:
            continue
        acc = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.sum() / n) * abs(acc - conf)
    return float(ece)


def calibration_report(y_test, prob_raw, prob_calibrated, tag=""):
    rows = []
    for name, p in [("raw", prob_raw), ("platt_calibrated", prob_calibrated)]:
        rows.append({
            "variant": name,
            "brier_score": brier_score_loss(y_test, p),
            "log_loss": log_loss(y_test, np.clip(p, 1e-6, 1 - 1e-6)),
            "ece": expected_calibration_error(y_test, p, CFG.ECE_BINS),
        })
    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_DIR / f"calibration_metrics{tag}.csv", index=False)

    fig, ax = plt.subplots(figsize=(6, 6))
    for name, p in [("Raw RF", prob_raw), ("Platt-calibrated", prob_calibrated)]:
        frac_pos, mean_pred = calibration_curve(y_test, p, n_bins=CFG.ECE_BINS, strategy="quantile")
        ax.plot(mean_pred, frac_pos, marker="o", label=name)
    ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed frequency")
    ax.set_title("Calibration Curve")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "calibration_curve.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    return df


# ================================================================
# 9. NEW -- FULL PREDICTIVE METRIC SUITE + BOOTSTRAP 95% CI
# ================================================================
def full_metric_suite(y_true, y_prob, y_pred) -> Dict[str, float]:
    y_true = np.asarray(y_true)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_ppv": precision_score(y_true, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "npv": npv,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "brier_score": brier_score_loss(y_true, y_prob),
        "log_loss": log_loss(y_true, np.clip(y_prob, 1e-6, 1 - 1e-6)),
    }


def bootstrap_metric_ci(y_true, y_prob, threshold, n_boot=None, seed=CFG.RANDOM_STATE) -> pd.DataFrame:
    n_boot = n_boot or CFG.N_BOOTSTRAP_CI
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    rng = np.random.RandomState(seed)
    n = len(y_true)
    records = []
    for b in range(n_boot):
        idx = rng.randint(0, n, n)
        yt, yp = y_true[idx], y_prob[idx]
        if len(np.unique(yt)) < 2:
            continue
        pred = (yp >= threshold).astype(int)
        records.append(full_metric_suite(yt, yp, pred))
    boot_df = pd.DataFrame(records)

    summary = []
    for col in boot_df.columns:
        vals = boot_df[col].dropna().values
        summary.append({
            "metric": col,
            "mean": float(np.mean(vals)),
            "ci_lower_2.5": float(np.percentile(vals, 2.5)),
            "ci_upper_97.5": float(np.percentile(vals, 97.5)),
        })
    summary_df = pd.DataFrame(summary)
    summary_df.to_csv(OUTPUT_DIR / "test_metrics_bootstrap_ci.csv", index=False)
    return summary_df


# ================================================================
# 10. NEW -- DECISION CURVE ANALYSIS
# ================================================================
def decision_curve_analysis(y_true, y_prob) -> pd.DataFrame:
    y_true = np.asarray(y_true)
    n = len(y_true)
    prevalence = y_true.mean()
    start, stop, step = CFG.DCA_THRESHOLDS
    thresholds = np.arange(start, stop + 1e-9, step)

    rows = []
    for pt in thresholds:
        pred = (y_prob >= pt).astype(int)
        tp = int(((pred == 1) & (y_true == 1)).sum())
        fp = int(((pred == 1) & (y_true == 0)).sum())
        odds = pt / (1 - pt)
        nb_model = (tp / n) - (fp / n) * odds
        nb_all = prevalence - (1 - prevalence) * odds
        nb_none = 0.0
        rows.append({"threshold": pt, "net_benefit_model": nb_model,
                      "net_benefit_treat_all": nb_all, "net_benefit_treat_none": nb_none})

    dca_df = pd.DataFrame(rows)
    dca_df.to_csv(OUTPUT_DIR / "decision_curve_analysis.csv", index=False)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(dca_df["threshold"], dca_df["net_benefit_model"], label="ACO-RF model", linewidth=2)
    ax.plot(dca_df["threshold"], dca_df["net_benefit_treat_all"], label="Treat all", linestyle="--")
    ax.plot(dca_df["threshold"], dca_df["net_benefit_treat_none"], label="Treat none", linestyle=":")
    ax.set_xlabel("Threshold probability")
    ax.set_ylabel("Net benefit")
    ax.set_title("Decision Curve Analysis")
    ax.set_ylim(bottom=-0.05)
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "decision_curve.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    return dca_df


# ================================================================
# 11. NEW -- BASELINE COMPARISON TABLE (expanded metric set)
# ================================================================
def run_baseline_comparison(X_train, y_train, X_test, y_test, selected_features, aco_model, aco_scaler, aco_threshold):
    print("\n" + "=" * 80)
    print("BASELINE COMPARISON (same protocol, expanded metrics)")
    print("=" * 80)

    models = {
        "LogisticRegression": LogisticRegression(max_iter=2000, class_weight="balanced"),
        "RandomForest_default": RandomForestClassifier(n_estimators=300, random_state=CFG.RANDOM_STATE, n_jobs=-1),
        "ExtraTrees": ExtraTreesClassifier(n_estimators=500, class_weight="balanced",
                                            random_state=CFG.RANDOM_STATE, n_jobs=-1),
        "GradientBoosting": GradientBoostingClassifier(random_state=CFG.RANDOM_STATE),
    }
    if HAS_XGB:
        models["XGBoost"] = XGBClassifier(
            n_estimators=300, eval_metric="logloss", random_state=CFG.RANDOM_STATE, n_jobs=-1
        )
    if HAS_LGBM:
        models["LightGBM"] = LGBMClassifier(n_estimators=300, random_state=CFG.RANDOM_STATE, n_jobs=-1, verbose=-1)

    scaler = MinMaxScaler()
    X_train_sel = X_train[selected_features]
    X_test_sel = X_test[selected_features]
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_sel), columns=selected_features, index=X_train_sel.index)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test_sel), columns=selected_features, index=X_test_sel.index)
    smote = SMOTE(random_state=CFG.RANDOM_STATE, k_neighbors=5)
    X_res, y_res = smote.fit_resample(X_train_scaled, y_train)

    rows = []
    for name, model in models.items():
        model.fit(X_res, y_res)
        prob = model.predict_proba(X_test_scaled)[:, 1]
        pred = (prob >= 0.50).astype(int)
        m = full_metric_suite(y_test, prob, pred)
        m["model"] = name
        rows.append(m)

    # Proposed ACO-RF (already fitted, using its own tuned threshold)
    prob_aco = aco_model.predict_proba(X_test[selected_features].pipe(
        lambda d: pd.DataFrame(aco_scaler.transform(d), columns=selected_features, index=d.index)
    ))[:, 1]
    pred_aco = (prob_aco >= aco_threshold).astype(int)
    m_aco = full_metric_suite(y_test, prob_aco, pred_aco)
    m_aco["model"] = "Proposed_ACO-RF (tuned threshold)"
    rows.append(m_aco)

    df = pd.DataFrame(rows).set_index("model")
    cols_order = ["accuracy", "balanced_accuracy", "f1", "mcc", "roc_auc", "pr_auc", "brier_score"]
    df_reported = df[cols_order]
    df_reported.to_csv(OUTPUT_DIR / "baseline_comparison_expanded.csv")
    print(df_reported.round(4).to_string())
    return df_reported


# ================================================================
# 12. NEW -- NESTED CROSS-VALIDATION ROBUSTNESS CHECK
# ================================================================
def nested_cv_robustness(X_train, y_train, selected_features):
    print("\n" + "=" * 80)
    print("NESTED CV ROBUSTNESS CHECK")
    print("=" * 80)

    X_sel = X_train[selected_features]
    outer_cv = StratifiedKFold(n_splits=CFG.NESTED_CV_OUTER_SPLITS, shuffle=True, random_state=CFG.RANDOM_STATE)

    param_grid = {
        "rf__n_estimators": list(CFG.N_ESTIMATORS_VALUES),
        "rf__max_depth": list(CFG.MAX_DEPTH_VALUES),
        "rf__min_samples_split": list(CFG.MIN_SAMPLES_SPLIT_VALUES),
        "rf__min_samples_leaf": list(CFG.MIN_SAMPLES_LEAF_VALUES),
        "rf__max_features": list(CFG.MAX_FEATURES_VALUES),
        "rf__class_weight": list(CFG.CLASS_WEIGHT_VALUES),
    }

    outer_scores = []
    for fold, (tr_idx, va_idx) in enumerate(outer_cv.split(X_sel, y_train), start=1):
        X_tr, X_va = X_sel.iloc[tr_idx], X_sel.iloc[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]
        inner_cv = StratifiedKFold(n_splits=CFG.INNER_CV_SPLITS, shuffle=True, random_state=CFG.RANDOM_STATE)

        if CFG.NESTED_CV_USE_FULL_ACO:
            aco_inner = ACORandomForest()
            best_params_inner, _ = aco_inner.optimize(X_tr, y_tr, verbose=False)
            best_pipeline = build_cv_pipeline(best_params_inner)
            best_pipeline.fit(X_tr, y_tr)
        else:
            pipeline = build_cv_pipeline({"n_estimators": 300})  # placeholder, overwritten by search
            search = RandomizedSearchCV(
                build_cv_pipeline({"n_estimators": 300}),
                param_distributions=param_grid,
                n_iter=CFG.NESTED_CV_RANDOM_ITER,
                scoring="roc_auc", cv=inner_cv, random_state=CFG.RANDOM_STATE, n_jobs=-1,
            )
            search.fit(X_tr, y_tr)
            best_pipeline = search.best_estimator_

        prob = best_pipeline.predict_proba(X_va)[:, 1]
        pred = (prob >= 0.50).astype(int)
        m = full_metric_suite(y_va, prob, pred)
        m["fold"] = fold
        outer_scores.append(m)
        print(f"  Outer fold {fold}: AUC={m['roc_auc']:.4f}  Acc={m['accuracy']:.4f}")

    nested_df = pd.DataFrame(outer_scores)
    nested_df.to_csv(OUTPUT_DIR / "nested_cv_results.csv", index=False)

    summary = nested_df.drop(columns=["fold"]).agg(["mean", "std"])
    summary.to_csv(OUTPUT_DIR / "nested_cv_summary.csv")
    print("\nNested CV summary (mean / std):")
    print(summary.round(4).to_string())
    if not CFG.NESTED_CV_USE_FULL_ACO:
        print(
            "\nNOTE: inner-loop tuning used RandomizedSearchCV "
            f"({CFG.NESTED_CV_RANDOM_ITER} draws) over the same search space as ACO, "
            "not a full independent ACO run per outer fold. Set "
            "CFG.NESTED_CV_USE_FULL_ACO=True for a stricter (much slower) check."
        )
    return nested_df, summary


# ================================================================
# 13. SHAP / LIME / COUNTERFACTUAL PRIMITIVES (unchanged core logic)
# ================================================================
def create_shap_explainer(model):
    return shap.TreeExplainer(model)


def get_shap_class1(explainer, X):
    values = explainer.shap_values(X)
    if isinstance(values, list):
        return np.asarray(values[1])
    values = np.asarray(values)
    if values.ndim == 3:
        if values.shape[-1] == 2:
            return values[:, :, 1]
        if values.shape[1] == 2:
            return values[:, 1, :]
    if values.ndim == 2:
        return values
    raise ValueError(f"Unexpected SHAP shape: {values.shape}")


def shap_direction(value):
    if value > 1e-8:
        return 1
    if value < -1e-8:
        return -1
    return 0


def extract_lime_direction(explanation, feature):
    for rule, weight in explanation.as_list():
        if feature in rule:
            weight = float(weight)
            if weight > 1e-8:
                return 1, weight, rule
            if weight < -1e-8:
                return -1, weight, rule
            return 0, weight, rule
    return 0, 0.0, "FEATURE_NOT_IN_LIME_TOP_K"


def detect_binary_features(X_scaled: pd.DataFrame, tol: float = 1e-6) -> Dict[str, bool]:
    """A MinMax-scaled feature that only ever takes values {0, 1} in the
    training data is a binary/one-hot clinical flag (e.g. MemoryComplaints).
    A fixed +-0.05 counterfactual step is not a meaningful perturbation for
    such features (0.95 is still 'mostly true'), so they need a full flip
    instead. Continuous features keep the small local step."""
    is_binary = {}
    for col in X_scaled.columns:
        uniq = X_scaled[col].dropna().unique()
        is_binary[col] = bool(len(uniq) <= 2 and np.all(np.isin(np.round(uniq, 6), [0.0, 1.0])))
    return is_binary


def counterfactual_direction(model, instance, feature, step=None, is_binary: Optional[bool] = None):
    step = step or CFG.COUNTERFACTUAL_STEP
    x = instance.copy()
    current = float(x[feature])

    if is_binary:
        # Full flip: the only counterfactual that is actually meaningful
        # for a binary clinical indicator.
        lower, upper = 0.0, 1.0
    else:
        lower = max(0.0, current - step)
        upper = min(1.0, current + step)

    if upper == lower:
        return 0, 0.0
    x_lower, x_upper = x.copy(), x.copy()
    x_lower[feature] = lower
    x_upper[feature] = upper
    p_lower = model.predict_proba(x_lower.to_frame().T)[0, 1]
    p_upper = model.predict_proba(x_upper.to_frame().T)[0, 1]
    delta = float(p_upper - p_lower)
    direction = 1 if delta > 1e-8 else -1 if delta < -1e-8 else 0
    return direction, delta


# ================================================================
# 14. NEW -- GLOBAL EXPLANATION CONSENSUS
#     (SHAP mean |value|  vs  RF impurity  vs  permutation importance)
# ================================================================
def global_explanation_consensus(model, shap_matrix, X_eval, y_eval, feature_names) -> Dict[str, Any]:
    shap_importance = pd.Series(np.abs(shap_matrix).mean(axis=0), index=feature_names)
    impurity_importance = pd.Series(model.feature_importances_, index=feature_names)

    perm = permutation_importance(
        model, X_eval, y_eval, n_repeats=20, random_state=CFG.RANDOM_STATE, scoring="roc_auc", n_jobs=-1,
    )
    perm_importance = pd.Series(perm.importances_mean, index=feature_names).clip(lower=0)

    rank_df = pd.DataFrame({
        "shap_importance": shap_importance,
        "impurity_importance": impurity_importance,
        "permutation_importance": perm_importance,
    })
    rank_df["shap_rank"] = rank_df["shap_importance"].rank(ascending=False)
    rank_df["impurity_rank"] = rank_df["impurity_importance"].rank(ascending=False)
    rank_df["permutation_rank"] = rank_df["permutation_importance"].rank(ascending=False)
    rank_df = rank_df.sort_values("shap_rank")
    rank_df.to_csv(OUTPUT_DIR / "global_consensus_importances.csv")

    pairs = [("shap_importance", "impurity_importance"),
             ("shap_importance", "permutation_importance"),
             ("impurity_importance", "permutation_importance")]
    spearman_results = {}
    for a, b in pairs:
        rho, p = spearmanr(rank_df[a], rank_df[b])
        spearman_results[f"spearman_{a}_vs_{b}"] = float(rho)

    jaccard_results = {}
    for k in CFG.TOP_K_CONSENSUS:
        top_sets = {
            "shap": set(rank_df.nsmallest(k, "shap_rank").index),
            "impurity": set(rank_df.nsmallest(k, "impurity_rank").index),
            "permutation": set(rank_df.nsmallest(k, "permutation_rank").index),
        }
        pair_jaccards = []
        for a, b in itertools.combinations(top_sets, 2):
            inter = len(top_sets[a] & top_sets[b])
            union = len(top_sets[a] | top_sets[b])
            pair_jaccards.append(inter / union if union else 0.0)
        jaccard_results[f"top{k}_jaccard_mean"] = float(np.mean(pair_jaccards))

    mean_spearman = float(np.mean(list(spearman_results.values())))
    mean_spearman_unit = (mean_spearman + 1) / 2  # rescale [-1,1] -> [0,1]
    mean_jaccard = float(np.mean(list(jaccard_results.values())))

    global_consensus_score = float(np.mean([mean_spearman_unit, mean_jaccard]))

    metrics = {**spearman_results, **jaccard_results,
               "mean_spearman_unit_interval": mean_spearman_unit,
               "mean_topk_jaccard": mean_jaccard,
               "global_consensus_score": global_consensus_score}
    pd.DataFrame([metrics]).to_csv(OUTPUT_DIR / "global_consensus_metrics.csv", index=False)

    fig, ax = plt.subplots(figsize=(8, 5))
    names = list(spearman_results.keys()) + list(jaccard_results.keys())
    vals = list(spearman_results.values()) + list(jaccard_results.values())
    ax.bar(range(len(names)), vals)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=35, ha="right")
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.set_title("Global Feature-Importance Consensus")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "global_consensus_plot.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    return {"rank_df": rank_df, "metrics": metrics, "global_consensus_score": global_consensus_score}


# ================================================================
# 15. NEW -- EXPLANATION STABILITY UNDER BOOTSTRAP RESAMPLING
# ================================================================
def explanation_stability(X_train_scaled, y_train, selected_features,
                           best_params, X_eval_fixed, original_shap_importance: pd.Series,
                           n_bootstrap=None, seed=CFG.RANDOM_STATE):
    """
    Retrain the final model on bootstrap resamples of the (pre-SMOTE) scaled
    training data, reapplying SMOTE inside each resample, then compute SHAP
    global importance on a FIXED evaluation set (X_eval_fixed = scaled test
    features) so that all bootstrap rankings are compared on a common basis.
    """
    n_bootstrap = n_bootstrap or CFG.N_BOOTSTRAP_STABILITY
    rng = np.random.RandomState(seed)
    n = len(X_train_scaled)

    boot_importances = []
    for b in range(n_bootstrap):
        idx = rng.randint(0, n, n)
        X_bs = X_train_scaled.iloc[idx].reset_index(drop=True)
        y_bs = y_train.iloc[idx].reset_index(drop=True)
        if len(np.unique(y_bs)) < 2:
            continue
        try:
            smote = SMOTE(random_state=seed + b, k_neighbors=5)
            X_res, y_res = smote.fit_resample(X_bs, y_bs)
        except ValueError:
            continue

        model_b = RandomForestClassifier(random_state=seed + b, n_jobs=-1, **best_params)
        model_b.fit(X_res, y_res)

        explainer_b = shap.TreeExplainer(model_b)
        shap_b = get_shap_class1(explainer_b, X_eval_fixed)
        imp_b = pd.Series(np.abs(shap_b).mean(axis=0), index=selected_features)
        signed_mean_b = pd.Series(shap_b.mean(axis=0), index=selected_features)
        boot_importances.append({"importance": imp_b, "signed_mean": signed_mean_b})

    if not boot_importances:
        raise RuntimeError("Explanation-stability bootstrap produced no valid resamples.")

    orig_rank = original_shap_importance.rank(ascending=False)
    orig_signed = original_shap_importance  # already |shap| mean; sign handled separately below

    spearman_list, jaccard5_list, jaccard10_list = [], [], []
    sign_matches = pd.Series(0, index=selected_features, dtype=float)

    for rec in boot_importances:
        imp_b = rec["importance"]
        rank_b = imp_b.rank(ascending=False)
        rho, _ = spearmanr(orig_rank, rank_b)
        spearman_list.append(rho)

        top5_orig = set(orig_rank.nsmallest(5).index)
        top5_b = set(rank_b.nsmallest(5).index)
        jaccard5_list.append(len(top5_orig & top5_b) / len(top5_orig | top5_b))

        top10_orig = set(orig_rank.nsmallest(10).index)
        top10_b = set(rank_b.nsmallest(10).index)
        jaccard10_list.append(len(top10_orig & top10_b) / len(top10_orig | top10_b))

    signed_matrix = pd.DataFrame([rec["signed_mean"] for rec in boot_importances])
    original_signed_direction = np.sign(signed_matrix.mean(axis=0))  # reference sign from bootstrap consensus
    sign_stability = (np.sign(signed_matrix) == original_signed_direction).mean(axis=0)

    mean_spearman = float(np.nanmean(spearman_list))
    mean_spearman_unit = (mean_spearman + 1) / 2
    mean_jaccard5 = float(np.mean(jaccard5_list))
    mean_jaccard10 = float(np.mean(jaccard10_list))
    mean_sign_stability = float(sign_stability.mean())

    stability_score = float(np.mean([mean_spearman_unit, np.mean([mean_jaccard5, mean_jaccard10]), mean_sign_stability]))

    per_feature = pd.DataFrame({
        "original_importance": original_shap_importance,
        "bootstrap_mean_importance": pd.concat([rec["importance"] for rec in boot_importances], axis=1).mean(axis=1),
        "bootstrap_std_importance": pd.concat([rec["importance"] for rec in boot_importances], axis=1).std(axis=1),
        "sign_stability": sign_stability,
    }).sort_values("original_importance", ascending=False)
    per_feature.to_csv(OUTPUT_DIR / "explanation_stability_per_feature.csv")

    summary = {
        "n_bootstrap_runs": len(boot_importances),
        "mean_spearman_rank_stability": mean_spearman,
        "mean_spearman_unit_interval": mean_spearman_unit,
        "mean_top5_jaccard_stability": mean_jaccard5,
        "mean_top10_jaccard_stability": mean_jaccard10,
        "mean_sign_stability": mean_sign_stability,
        "stability_score": stability_score,
    }
    pd.DataFrame([summary]).to_csv(OUTPUT_DIR / "explanation_stability_summary.csv", index=False)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.hist(spearman_list, bins=15)
    ax.set_xlabel("Spearman rank correlation vs. original SHAP ranking")
    ax.set_ylabel("Bootstrap runs")
    ax.set_title("Explanation Stability: Spearman Distribution")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "explanation_stability_plot.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    return {"summary": summary, "per_feature": per_feature, "stability_score": stability_score}


# ================================================================
# 16. LOCAL EXPLANATION CONSISTENCY (generalised XDCR: top-K features,
#     not just FunctionalAssessment)
# ================================================================
def local_explanation_consistency(model, shap_explainer, lime_explainer, X_eval, top_features,
                                   binary_feature_map: Optional[Dict[str, bool]] = None):
    binary_feature_map = binary_feature_map or {}
    shap_matrix = get_shap_class1(shap_explainer, X_eval)
    predictions = model.predict(X_eval)
    probabilities = model.predict_proba(X_eval)[:, 1]

    all_rows = []
    for pos in range(len(X_eval)):
        instance = X_eval.iloc[pos]
        lime_exp = lime_explainer.explain_instance(
            instance.values, model.predict_proba,
            num_features=max(CFG.LIME_NUM_FEATURES, len(top_features)),
            num_samples=CFG.LIME_NUM_SAMPLES,
        )
        for feature in top_features:
            feature_index = list(X_eval.columns).index(feature)
            shap_value = float(shap_matrix[pos, feature_index])
            shap_dir = shap_direction(shap_value)

            lime_dir, lime_weight, lime_rule = extract_lime_direction(lime_exp, feature)
            cf_dir, cf_delta = counterfactual_direction(
                model, instance, feature, is_binary=binary_feature_map.get(feature, False)
            )

            valid_three = shap_dir != 0 and lime_dir != 0 and cf_dir != 0
            three_agree = valid_three and shap_dir == lime_dir == cf_dir

            all_rows.append({
                "test_index": X_eval.index[pos], "feature": feature,
                "prediction": int(predictions[pos]), "probability": float(probabilities[pos]),
                "SHAP_dir": shap_dir, "LIME_dir": lime_dir, "CF_dir": cf_dir,
                "three_way_comparable": bool(valid_three), "three_way_agree": bool(three_agree),
                "severity": abs(shap_value) + abs(lime_weight) + abs(cf_delta),
            })

    results = pd.DataFrame(all_rows)
    results.to_csv(OUTPUT_DIR / "local_consistency_instance_level.csv", index=False)

    per_feature = results.groupby("feature").apply(
        lambda g: pd.Series({
            "n_comparable": g["three_way_comparable"].sum(),
            "three_way_agreement_rate": g.loc[g["three_way_comparable"], "three_way_agree"].mean()
            if g["three_way_comparable"].sum() > 0 else np.nan,
        })
    )
    per_feature.to_csv(OUTPUT_DIR / "local_consistency_per_feature.csv")

    local_reliability_score = float(per_feature["three_way_agreement_rate"].dropna().mean())

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(per_feature.index, per_feature["three_way_agreement_rate"] * 100)
    ax.set_ylabel("Three-way agreement rate (%)")
    ax.set_title("Local Explanation Consistency (SHAP vs LIME vs Counterfactual)")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "local_consistency_plot.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    return {"instance_level": results, "per_feature": per_feature, "local_reliability_score": local_reliability_score}


# ================================================================
# 17. DOMAIN-LEVEL SHAP ATTRIBUTION
# ================================================================
def domain_attribution(shap_matrix, feature_names):
    importance = pd.Series(np.abs(shap_matrix).mean(axis=0), index=feature_names)
    domain_totals = {}
    for feat, val in importance.items():
        d = feature_domain(feat)
        domain_totals[d] = domain_totals.get(d, 0.0) + val
    total = sum(domain_totals.values())
    domain_pct = {k: 100 * v / total for k, v in domain_totals.items()}
    df = pd.DataFrame({"domain": list(domain_pct.keys()), "pct_contribution": list(domain_pct.values())})
    df = df.sort_values("pct_contribution", ascending=False)
    df.to_csv(OUTPUT_DIR / "domain_attribution.csv", index=False)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(df["domain"], df["pct_contribution"])
    ax.set_ylabel("% of total |SHAP| attribution")
    ax.set_title("Domain-Level SHAP Attribution")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "domain_attribution_plot.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    return df


# ================================================================
# 18. TERS -- TRIANGULATED EXPLAINABILITY RELIABILITY SCORE
#     Proposed methodological metric. NOT medically validated.
#     Requires external-dataset validation before any clinical framing.
# ================================================================
def compute_ters(global_consensus_score, stability_score, local_reliability_score,
                  weights: Optional[Dict[str, float]] = None):
    weights = dict(weights or CFG.TERS_WEIGHTS)
    components = {"global": global_consensus_score, "stability": stability_score,
                  "local": local_reliability_score}

    # If a component is NaN (e.g. too few comparable local-consistency
    # instances for the chosen top-K features), drop it and renormalise the
    # remaining weights rather than silently propagating NaN into TERS.
    valid = {k: v for k, v in components.items() if v is not None and not (isinstance(v, float) and np.isnan(v))}
    if not valid:
        return float("nan")
    if len(valid) < len(components):
        dropped = set(components) - set(valid)
        print(f"WARNING: TERS component(s) {dropped} were NaN and excluded; "
              "weights renormalised over the remaining components.")
    w_sum = sum(weights[k] for k in valid)
    ters = sum(weights[k] * components[k] for k in valid) / w_sum
    return float(ters)


def ters_sensitivity_analysis(global_consensus_score, stability_score, local_reliability_score):
    weight_schemes = {
        "equal_weight": {"global": 1/3, "stability": 1/3, "local": 1/3},
        "stability_heavy": {"global": 0.2, "stability": 0.6, "local": 0.2},
        "local_heavy": {"global": 0.2, "stability": 0.2, "local": 0.6},
        "global_heavy": {"global": 0.6, "stability": 0.2, "local": 0.2},
    }
    rows = []
    for name, w in weight_schemes.items():
        ters = compute_ters(global_consensus_score, stability_score, local_reliability_score, w)
        rows.append({"weight_scheme": name, **w, "TERS": ters})
    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_DIR / "TERS_sensitivity_analysis.csv", index=False)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(df["weight_scheme"], df["TERS"])
    ax.set_ylim(0, 1)
    ax.set_ylabel("TERS")
    ax.set_title("TERS Sensitivity to Component Weighting")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "TERS_component_plot.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    return df


# ================================================================
# 19. ROBUSTNESS EXPERIMENTS: seed stability / ablations / SMOTE toggle
# ================================================================
def seed_stability_check(X_train, y_train, selected_features, best_params, n_repeats=None):
    n_repeats = n_repeats or CFG.N_SEED_REPEATS
    rows = []
    for seed in range(n_repeats):
        cv = StratifiedKFold(n_splits=CFG.CV_SPLITS, shuffle=True, random_state=seed)
        params = dict(best_params)
        pipeline = build_cv_pipeline(params)
        scores = cross_validate(pipeline, X_train[selected_features], y_train, cv=cv,
                                 scoring={"accuracy": "accuracy", "f1": "f1", "auc": "roc_auc"}, n_jobs=1)
        rows.append({"seed": seed, "accuracy": scores["test_accuracy"].mean(),
                     "f1": scores["test_f1"].mean(), "auc": scores["test_auc"].mean()})
    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_DIR / "seed_stability.csv", index=False)
    summary = df[["accuracy", "f1", "auc"]].agg(["mean", "std"])
    summary.to_csv(OUTPUT_DIR / "seed_stability_summary.csv")
    return df, summary


def cognitive_feature_ablation(X_train, y_train, selected_features, best_params):
    cognitive_like = [f for f in selected_features if feature_domain(f) == "cognitive_functional"]
    reduced_features = [f for f in selected_features if f not in cognitive_like]

    cv = StratifiedKFold(n_splits=CFG.CV_SPLITS, shuffle=True, random_state=CFG.RANDOM_STATE)
    rows = []
    for tag, feats in [("full_feature_set", selected_features), ("without_cognitive_functional", reduced_features)]:
        if len(feats) == 0:
            continue
        pipeline = build_cv_pipeline(best_params)
        scores = cross_validate(pipeline, X_train[feats], y_train, cv=cv,
                                 scoring={"accuracy": "accuracy", "f1": "f1", "auc": "roc_auc"}, n_jobs=1)
        rows.append({"variant": tag, "n_features": len(feats),
                     "cv_accuracy_mean": scores["test_accuracy"].mean(),
                     "cv_f1_mean": scores["test_f1"].mean(),
                     "cv_auc_mean": scores["test_auc"].mean()})
    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_DIR / "cognitive_ablation.csv", index=False)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(df["variant"], df["cv_auc_mean"])
    ax.set_ylabel("CV ROC-AUC")
    ax.set_title("Ablation: Cognitive/Functional Features Removed")
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "ablation_plot.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    return df


def smote_sensitivity(X_train, y_train, selected_features, best_params):
    cv = StratifiedKFold(n_splits=CFG.CV_SPLITS, shuffle=True, random_state=CFG.RANDOM_STATE)
    rows = []
    for tag, use_smote in [("with_SMOTE", True), ("without_SMOTE", False)]:
        pipeline = build_cv_pipeline(best_params, use_smote=use_smote)
        scores = cross_validate(pipeline, X_train[selected_features], y_train, cv=cv,
                                 scoring={"accuracy": "accuracy", "balanced_accuracy": "balanced_accuracy",
                                          "f1": "f1", "auc": "roc_auc"}, n_jobs=1)
        rows.append({"variant": tag,
                     "cv_accuracy_mean": scores["test_accuracy"].mean(),
                     "cv_balanced_accuracy_mean": scores["test_balanced_accuracy"].mean(),
                     "cv_f1_mean": scores["test_f1"].mean(),
                     "cv_auc_mean": scores["test_auc"].mean()})
    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_DIR / "smote_sensitivity.csv", index=False)
    return df


# ================================================================
# 20. SAVE ARTIFACTS + FINAL RESEARCH SUMMARY
# ================================================================
def save_artifacts(model, scaler, calibrator, selected_features, best_params, threshold, aco_history):
    joblib.dump(model, OUTPUT_DIR / "ACO_RF_model.pkl")
    joblib.dump(scaler, OUTPUT_DIR / "MinMax_scaler.pkl")
    joblib.dump(calibrator, OUTPUT_DIR / "Platt_calibrator.pkl")
    pd.DataFrame({"feature": selected_features}).to_csv(OUTPUT_DIR / "selected_features.csv", index=False)
    pd.DataFrame([best_params]).to_csv(OUTPUT_DIR / "ACO_best_parameters.csv", index=False)
    pd.DataFrame({"threshold": [threshold]}).to_csv(OUTPUT_DIR / "optimized_threshold.csv", index=False)
    aco_history.to_csv(OUTPUT_DIR / "ACO_convergence_history.csv", index=False)


def write_research_summary(context: Dict[str, Any]):
    lines = ["# Final Research Summary\n"]
    for k, v in context.items():
        lines.append(f"## {k}\n{v}\n")
    (OUTPUT_DIR / "final_research_summary.md").write_text("\n".join(lines))


# ================================================================
# 21. MAIN PIPELINE
# ================================================================
def main():
    df = load_dataset()
    X, y = prepare_data(df)
    X_train, X_test, y_train, y_test = split_data(X, y)

    selected_features, selector = run_befs(X_train, y_train)

    aco = ACORandomForest()
    best_params, aco_history = aco.optimize(X_train[selected_features], y_train)

    threshold, oof_prob, threshold_df = optimize_threshold(X_train, y_train, selected_features, best_params)

    (model, scaler, X_train_scaled, X_test_scaled,
     X_resampled, y_resampled) = fit_final_model(X_train, y_train, X_test, selected_features, best_params)

    # ---- Calibration (leakage-safe: fit on OOF, apply to test) ----
    calibrator = fit_platt_calibrator(oof_prob, y_train)
    prob_test_raw = model.predict_proba(X_test_scaled)[:, 1]
    prob_test_calibrated = apply_calibrator(calibrator, prob_test_raw)
    calibration_report(y_test, prob_test_raw, prob_test_calibrated)

    # ---- Full metrics + bootstrap CI (on calibrated probabilities) ----
    pred_test = (prob_test_calibrated >= threshold).astype(int)
    test_metrics = full_metric_suite(y_test, prob_test_calibrated, pred_test)
    pd.DataFrame([test_metrics]).to_csv(OUTPUT_DIR / "test_metrics_full.csv", index=False)
    ci_df = bootstrap_metric_ci(y_test, prob_test_calibrated, threshold)
    print("\nFinal test classification report (tuned + calibrated threshold):")
    print(classification_report(y_test, pred_test, target_names=["No Alzheimer", "Alzheimer"], digits=4))

    cm = confusion_matrix(y_test, pred_test)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=["No Alzheimer", "Alzheimer"],
                yticklabels=["No Alzheimer", "Alzheimer"], ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Confusion Matrix - Proposed ACO-RF")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    # ---- Decision curve analysis ----
    dca_df = decision_curve_analysis(y_test, prob_test_calibrated)

    # ---- Baseline comparison table ----
    baseline_df = run_baseline_comparison(X_train, y_train, X_test, y_test, selected_features,
                                           model, scaler, threshold)

    # ---- Nested CV robustness ----
    nested_df, nested_summary = nested_cv_robustness(X_train, y_train, selected_features)

    # ---- SHAP on test set ----
    shap_explainer = create_shap_explainer(model)
    shap_matrix = get_shap_class1(shap_explainer, X_test_scaled)
    original_shap_importance = pd.Series(np.abs(shap_matrix).mean(axis=0), index=selected_features)

    # ---- Global consensus (SHAP vs impurity vs permutation) ----
    consensus = global_explanation_consensus(model, shap_matrix, X_test_scaled, y_test, selected_features)

    # ---- Explanation stability (bootstrap retrain) ----
    stability = explanation_stability(
        X_train_scaled, y_train, selected_features, best_params,
        X_test_scaled, original_shap_importance,
    )

    # ---- Local explanation consistency on top-K features ----
    top_k_features = original_shap_importance.sort_values(ascending=False).head(CFG.TOP_K_LOCAL_FEATURES).index.tolist()
    lime_explainer = LimeTabularExplainer(
        training_data=X_train_scaled.values, feature_names=selected_features,
        class_names=["No Alzheimer", "Alzheimer"], mode="classification",
        discretize_continuous=True, random_state=CFG.RANDOM_STATE,
    )
    X_local_eval = X_test_scaled.iloc[:CFG.XAI_MAX_INSTANCES].copy()
    binary_feature_map = detect_binary_features(X_train_scaled)
    local = local_explanation_consistency(
        model, shap_explainer, lime_explainer, X_local_eval, top_k_features, binary_feature_map
    )

    # ---- Domain attribution ----
    domain_df = domain_attribution(shap_matrix, selected_features)

    # ---- TERS ----
    ters = compute_ters(consensus["global_consensus_score"], stability["stability_score"], local["local_reliability_score"])
    ters_sens = ters_sensitivity_analysis(consensus["global_consensus_score"], stability["stability_score"], local["local_reliability_score"])
    pd.DataFrame([{
        "global_consensus_score": consensus["global_consensus_score"],
        "stability_score": stability["stability_score"],
        "local_reliability_score": local["local_reliability_score"],
        "TERS": ters,
    }]).to_csv(OUTPUT_DIR / "TERS_final.csv", index=False)

    # ---- Robustness experiments ----
    seed_df, seed_summary = seed_stability_check(X_train, y_train, selected_features, best_params)
    ablation_df = cognitive_feature_ablation(X_train, y_train, selected_features, best_params)
    smote_df = smote_sensitivity(X_train, y_train, selected_features, best_params)

    # ---- Save artifacts ----
    save_artifacts(model, scaler, calibrator, selected_features, best_params, threshold, aco_history)

    write_research_summary({
        "dataset": "El-Kharoua-style Alzheimer's dataset (see DATA_URL)",
        "n_selected_features": len(selected_features),
        "aco_best_params": json.dumps(best_params),
        "tuned_threshold": threshold,
        "test_metrics": json.dumps(test_metrics),
        "global_consensus_score": consensus["global_consensus_score"],
        "stability_score": stability["stability_score"],
        "local_reliability_score": local["local_reliability_score"],
        "TERS": ters,
    })

    print("\nDONE. All outputs saved to:", OUTPUT_DIR.resolve())


if __name__ == "__main__":
    main()


LOADING ALZHEIMER DATASET
Shape: (2149, 35)
Diagnosis
0    1389
1     760
Name: count, dtype: int64
Predictor count after candidate feature engineering: 37
Train: (1504, 37)  Test: (645, 37)

BACKWARD FEATURE ELIMINATION (LEAKAGE-SAFE)
Selected 26 features.
Iter 01/20 | Acc=94.88% F1=92.64% AUC=0.9519 Global=0.9427
Iter 02/20 | Acc=95.15% F1=93.04% AUC=0.9525 Global=0.9454
Iter 03/20 | Acc=95.15% F1=93.04% AUC=0.9525 Global=0.9454
Iter 04/20 | Acc=95.15% F1=93.04% AUC=0.9525 Global=0.9454
Iter 05/20 | Acc=95.15% F1=93.04% AUC=0.9525 Global=0.9454
Iter 06/20 | Acc=95.15% F1=93.04% AUC=0.9525 Global=0.9454
Iter 07/20 | Acc=95.15% F1=93.04% AUC=0.9525 Global=0.9454
Iter 08/20 | Acc=95.15% F1=93.04% AUC=0.9525 Global=0.9454
Iter 09/20 | Acc=95.15% F1=93.04% AUC=0.9525 Global=0.9454
Iter 10/20 | Acc=95.15% F1=93.04% AUC=0.9525 Global=0.9454
Iter 11/20 | Acc=95.15% F1=93.04% AUC=0.9525 Global=0.9454
Iter 12/20 | Acc=95.15% F1=93.04% AUC=0.9525 Global=0.9454
Iter 13/20 | Acc=95.15% F1=93.04% 

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("/content/outputs_ters")

print("=" * 60)
print("DONE — ALL OUTPUTS SAVED")
print("=" * 60)
print(f"Location: {OUTPUT_DIR}")
print()

files = sorted(OUTPUT_DIR.rglob("*"))

for f in files:
    if f.is_file():
        print(f"✓ {f.relative_to(OUTPUT_DIR)}")

print()
print(f"Total files: {sum(f.is_file() for f in files)}")
print("=" * 60)

DONE — ALL OUTPUTS SAVED
Location: /content/outputs_ters

✓ ACO_RF_model.pkl
✓ ACO_best_parameters.csv
✓ ACO_convergence_history.csv
✓ MinMax_scaler.pkl
✓ Platt_calibrator.pkl
✓ TERS_component_plot.png
✓ TERS_final.csv
✓ TERS_sensitivity_analysis.csv
✓ ablation_plot.png
✓ baseline_comparison_expanded.csv
✓ calibration_curve.png
✓ calibration_metrics.csv
✓ cognitive_ablation.csv
✓ confusion_matrix.png
✓ decision_curve.png
✓ decision_curve_analysis.csv
✓ domain_attribution.csv
✓ domain_attribution_plot.png
✓ explanation_stability_per_feature.csv
✓ explanation_stability_plot.png
✓ explanation_stability_summary.csv
✓ final_research_summary.md
✓ global_consensus_importances.csv
✓ global_consensus_metrics.csv
✓ global_consensus_plot.png
✓ local_consistency_instance_level.csv
✓ local_consistency_per_feature.csv
✓ local_consistency_plot.png
✓ nested_cv_results.csv
✓ nested_cv_summary.csv
✓ optimized_threshold.csv
✓ seed_stability.csv
✓ seed_stability_summary.csv
✓ selected_features.csv
✓ smote

In [ ]:
!zip -r /content/outputs_ters.zip /content/outputs_ters

  adding: content/outputs_ters/ (stored 0%)
  adding: content/outputs_ters/baseline_comparison_expanded.csv (deflated 43%)
  adding: content/outputs_ters/domain_attribution_plot.png (deflated 19%)
  adding: content/outputs_ters/domain_attribution.csv (deflated 21%)
  adding: content/outputs_ters/smote_sensitivity.csv (deflated 36%)
  adding: content/outputs_ters/global_consensus_metrics.csv (deflated 51%)
  adding: content/outputs_ters/local_consistency_instance_level.csv (deflated 76%)
  adding: content/outputs_ters/global_consensus_importances.csv (deflated 49%)
  adding: content/outputs_ters/cognitive_ablation.csv (deflated 31%)
  adding: content/outputs_ters/seed_stability.csv (deflated 50%)
  adding: content/outputs_ters/decision_curve_analysis.csv (deflated 59%)
  adding: content/outputs_ters/MinMax_scaler.pkl (deflated 46%)
  adding: content/outputs_ters/nested_cv_summary.csv (deflated 43%)
  adding: content/outputs_ters/local_consistency_per_feature.csv (deflated 21%)
  adding: